In [ ]:
# Cell 1: package imports and portable data/output locations.
import os
import json
import math
import time
import warnings
from pathlib import Path
from dataclasses import asdict, replace
from itertools import product
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from bcke import *
from bcke.implementation import _validate_ordered_scalar_comparison, _bcke_vtu_state

PROJECT_ROOT = Path(os.environ.get("BCKE_PROJECT_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file() and (PROJECT_ROOT.parent / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("BCKE_DATA_DIR", PROJECT_ROOT / "data"))
OUTPUT_DIR = Path(os.environ.get("BCKE_OUTPUT_DIR", PROJECT_ROOT / "outputs"))
CYLINDER_VTU_ROOT = Path(os.environ.get("BCKE_CYLINDER_VTU_ROOT", DATA_DIR / "cylinder" / "vtu"))
STEP_VTU_ROOT = Path(os.environ.get("BCKE_STEP_VTU_ROOT", DATA_DIR / "step" / "vtu"))


In [ ]:
# Cell 2: cylinder BCKE experiment with full-sample reconstruction.
# Run Cell 1 before this cell. Results use the full_sample_ prefix.
# Run Cell 1 once; after changing settings, rerun only this cell.
# PyTorch supplies matrix operations and automatic differentiation, NOT a neural network.

# ============================================================================
# A. DATA AND EXPERIMENT PROTOCOL
# ============================================================================
FULL_SAMPLE_DATA_SOURCE = "npz"                # "ppmd", "npz", or "synthetic"
FULL_SAMPLE_PPMD_MODULE_DIRECTORY = None  # folder containing Diffusion.py
FULL_SAMPLE_STATE_COMPONENT = "u"              # same u-only state as the previous notebook; "uv" uses both
FULL_SAMPLE_NPZ_DATA_PATH = (DATA_DIR / "cylinder.npz")  # keys: states (N,Nh), parameters (N,q); optional weights, case_ids
FULL_SAMPLE_WEIGHTS_PATH = None                # .npy diagonal mass weights or full SPD W; None uses identity
FULL_SAMPLE_PARAMETER_FILE = None              # optional .npy array of TRUE input parameters in snapshot order

# Legacy viscosity grid from the old notebook, NOT uniformly spaced Reynolds numbers.
# Replace this with the actual labels used in your simulation, or set PARAMETER_FILE.
# For a genuinely regenerated Re=50,100,...,5000 dataset in that order, use:
# PARAMETER_VALUES = np.arange(50.0, 5000.0 + 50.0, 50.0)
FULL_SAMPLE_PARAMETER_VALUES = np.linspace(0.2e-4, 10.1e-4, 100)
FULL_SAMPLE_CASE_IDS = None                    # optional physical-case labels; all times of a case stay together

FULL_SAMPLE_N_TOTAL = 100                      # complete parameter cases, not spatial degrees of freedom
FULL_SAMPLE_N_REPR_TRAIN = 80
FULL_SAMPLE_N_FINAL_FIT = 90                   # 80 train + 10 validation + 10 untouched test
FULL_SAMPLE_SPLIT_MODE = "ordered"             # file order; "random" gives seeded case-wise interpolation split
FULL_SAMPLE_REFIT_ON_TRAIN_VALIDATION = True

FULL_SAMPLE_QUICK_TEST = False                  # True is a pipeline check, NOT a converged publication run
FULL_SAMPLE_QUICK_ITERATIONS = 30
FULL_SAMPLE_FULL_ITERATIONS = 600
FULL_SAMPLE_TORCH_THREADS = 1                  # small dense sample-space systems often benefit from one thread

# ============================================================================
# B. RECONSTRUCTION: CORE BCKE PARAMETERS (notation follows the LaTeX)
# ============================================================================
full_sample_reconstruction_cfg = BCKEFullSampleConfig(
    p=7, d=5, max_total_dimension=12,
    k_start=10,
    virtual_fractions=(0.2, 0.4, 0.6, 0.8),
    lambda_phi=5e-3,                # coordinate-map ridge, lambda_Phi
    lambda_lambda=8e-5,             # residual-map ridge, lambda_Lambda
    sigma_phi_factor=1,           # times the fixed residual distance quantile
    sigma_lambda_factor=0.5,        # times the initial coordinate distance quantile
    eta_a=1.0, alpha_x=1.0, alpha_z=1.0, beta_f=1.0, beta_b=1.0,
    omega_delta=0.01,
    gamma_diff=0.0,                 # 0: core BCKE; >0: the separate BCKE-D extension
    allow_ablation=True,           # only enable for explicitly labelled zero-weight comparisons
    distance_quantile=0.7,
    affinity_factors=(1.0, 1.0, 1.0, 1.0),  # epsilon_X, epsilon_hatZ, epsilon_V, epsilon_LambdaV
    bandwidth_fallback=0.5,
    eps_scale=1e-12,
    degree_safe=1e-14,
    bandwidth_degree_floor=1e-8,
    bandwidth_growth=1.2,
    max_bandwidth_expansions=200,
    rank_eigenvalue_rtol=1e-12,
    residual_zero_rtol=0.0,
    armijo_c=1e-4, armijo_rho=0.9, initial_step=1.0,
    grad_tol=1e-7, max_backtracks=1000,
    verbose_every=100,
    diff_chunk_size=64,
    seed=42, device="cpu",          # float64 throughout; CUDA requires a matching PyTorch installation
)

# One candidate per quantity is the default for a manageable first run.
# Expand these lists for the paper's finite-grid validation selection.
# This grid OVERRIDES the corresponding six values in reconstruction_cfg.
full_sample_representation_grid = dict(
    p=(full_sample_reconstruction_cfg.p,),
    d=(full_sample_reconstruction_cfg.d,),
    lambda_phi=(full_sample_reconstruction_cfg.lambda_phi,),
    lambda_lambda=(full_sample_reconstruction_cfg.lambda_lambda,),
    sigma_phi_factor=(full_sample_reconstruction_cfg.sigma_phi_factor,),
    sigma_lambda_factor=(full_sample_reconstruction_cfg.sigma_lambda_factor,),
)

# ============================================================================
# C. PREDICTION: BLOCK PARAMETER KRR + DECODER-AWARE RESIDUAL CORRECTION
# ============================================================================
full_sample_prediction_cfg = PredictionConfig(
    # The old shared Gaussian grid below remains an exact, zero-mean baseline.
    # Each pair is (linear a block, nonlinear z block); a is ignored when p=0.
    block_pairs=(
        (KernelBlockConfig("gaussian", 1.0, 1e-4), KernelBlockConfig("gaussian", 0.5, 1e-2)),
        (KernelBlockConfig("gaussian", 2.0, 1e-4), KernelBlockConfig("matern52", 1.0, 1e-4)),
        (KernelBlockConfig("matern52", 1.0, 1e-4), KernelBlockConfig("matern52", 0.3, 1e-4)),
    ),
    # For multi-parameter anisotropy, add fixed pairs with axis_factors=(...).
    # Values are positive multipliers, one per input column; choose before validation.
    correction_nus=(0.1, 0.5),  # B=0 is always included separately
    correction_max_iterations=30 if FULL_SAMPLE_QUICK_TEST else 200,
    correction_grad_tol=1e-7,
    correction_initial_step=1.0,
    correction_armijo_c=1e-4, correction_armijo_rho=0.8,
    correction_max_backtracks=40, correction_verbose_every=0,
    sigma_factors=(0.5, 1.0, ),
    ridges=(1e-8, 1e-6, ),  # one value also works: ridges=1e-4 or (1e-4,)
    distance_quantile=0.6,
    bandwidth_fallback=1.0,
)
# Every candidate uses [A; hat Z]. Separate a/z kernels are allowed.
# Only B is optimized in the correction; the BCKE decoder stays frozen.
# Joint selection uses full-state validation prediction error.

# Forecast settings for the cylinder experiment.
# All definitions are already in Cell 1; no later cell is required.
FULL_SAMPLE_CELL2_DMD_WINDOW = 7
FULL_SAMPLE_CELL2_DMD_RIDGE = 1e-05
FULL_SAMPLE_CELL2_DMD_VARIATION_RIDGE = 0.008
FULL_SAMPLE_CELL2_DMD_MAX_STEPS = 2000
FULL_SAMPLE_CELL2_DECODER_WINDOW = 3
FULL_SAMPLE_CELL2_DECODER_RIDGE = 1e-05
FULL_SAMPLE_CELL2_DECODER_MAX_HORIZON_STEPS = 10
FULL_SAMPLE_CELL2_POLY_CV_ORIGINS = (80,)
FULL_SAMPLE_CELL2_POLY_CV_HORIZON = 10
FULL_SAMPLE_CELL2_POLY_DEGREES = (3,)
FULL_SAMPLE_CELL2_POLY_WINDOWS = (9,)
FULL_SAMPLE_CELL2_POLY_RIDGES = (4e-05,)
FULL_SAMPLE_CELL2_POLY_TERMINAL_WEIGHT = 0.5

FULL_SAMPLE_CHECK_DIRECTIONAL_DERIVATIVES = True
FULL_SAMPLE_DIRECTIONAL_H = 1e-6
FULL_SAMPLE_DIRECTIONAL_TOLERANCE = 5e-4
FULL_SAMPLE_SAVE_OUTPUTS = True
FULL_SAMPLE_OUTPUT_ROOT = (OUTPUT_DIR / "bcke_decoder_aware_hybrid_outputs")
FULL_SAMPLE_FIGURE_DPI = 220

# Only the NEW figures use these axis settings; they do not affect fitting.
FULL_SAMPLE_PARAMETER_AXIS = 0  # zero-based input column; a scalar parameter uses 0
FULL_SAMPLE_PARAMETER_AXIS_LABEL = r"parameter $\mu$"  # for Reynolds data: "Reynolds number Re"

# ============================================================================
# D. LOAD DATA AND RUN THE COMPLETE PROTOCOL
# ============================================================================
torch.set_num_threads(FULL_SAMPLE_TORCH_THREADS)
torch.manual_seed(full_sample_reconstruction_cfg.seed)
np.random.seed(full_sample_reconstruction_cfg.seed)
if FULL_SAMPLE_DATA_SOURCE == "ppmd":
    full_sample_fields_all = load_ppmd_states(FULL_SAMPLE_STATE_COMPONENT, FULL_SAMPLE_PPMD_MODULE_DIRECTORY)
    full_sample_mu_full = np.load(FULL_SAMPLE_PARAMETER_FILE, allow_pickle=False) if FULL_SAMPLE_PARAMETER_FILE is not None else FULL_SAMPLE_PARAMETER_VALUES
    full_sample_weight_matrix = None if FULL_SAMPLE_WEIGHTS_PATH is None else np.load(FULL_SAMPLE_WEIGHTS_PATH, allow_pickle=False)
    print("Check: parameter labels must exactly match the order of the loaded snapshots.")
    print("The default labels are the OLD viscosity grid; they do not describe a new uniform-Re dataset.")
elif FULL_SAMPLE_DATA_SOURCE == "npz":
    with np.load(FULL_SAMPLE_NPZ_DATA_PATH, allow_pickle=False) as full_sample_dataset:
        full_sample_fields_all, full_sample_mu_full = full_sample_dataset["states"], full_sample_dataset["parameters"]
        full_sample_weight_matrix = full_sample_dataset["weights"] if "weights" in full_sample_dataset else None
        FULL_SAMPLE_CASE_IDS = full_sample_dataset["case_ids"] if "case_ids" in full_sample_dataset else FULL_SAMPLE_CASE_IDS
    if FULL_SAMPLE_WEIGHTS_PATH is not None:
        full_sample_weight_matrix = np.load(FULL_SAMPLE_WEIGHTS_PATH, allow_pickle=False)
elif FULL_SAMPLE_DATA_SOURCE == "synthetic":
    # Explicit lightweight smoke-test configuration; no CFD data is read.
    full_sample_fields_all, full_sample_mu_full, full_sample_weight_matrix = synthetic_bcke_data(n=24, nh=14, seed=full_sample_reconstruction_cfg.seed)
    FULL_SAMPLE_N_TOTAL, FULL_SAMPLE_N_REPR_TRAIN, FULL_SAMPLE_N_FINAL_FIT = 24, 16, 20
    FULL_SAMPLE_CELL2_POLY_CV_ORIGINS, FULL_SAMPLE_CELL2_POLY_CV_HORIZON = (16,), 4
    full_sample_reconstruction_cfg = replace(full_sample_reconstruction_cfg, p=2, d=2, max_total_dimension=4, k_start=3)
    full_sample_representation_grid = dict(p=(2,), d=(2,), lambda_phi=(full_sample_reconstruction_cfg.lambda_phi,),
        lambda_lambda=(full_sample_reconstruction_cfg.lambda_lambda,),
        sigma_phi_factor=(full_sample_reconstruction_cfg.sigma_phi_factor,),
        sigma_lambda_factor=(full_sample_reconstruction_cfg.sigma_lambda_factor,))
    FULL_SAMPLE_CASE_IDS = None
    print("SYNTHETIC SMOKE TEST: 24 samples, p=2, d=2; not cylinder-flow results.")
else:
    raise ValueError("Unknown DATA_SOURCE.")

full_sample_mu_full = parameter_array(full_sample_mu_full)
FULL_SAMPLE_MAX_ITERATIONS = FULL_SAMPLE_QUICK_ITERATIONS if FULL_SAMPLE_QUICK_TEST else FULL_SAMPLE_FULL_ITERATIONS
print("States:", full_sample_fields_all.shape, "parameters:", full_sample_mu_full.shape)
print("W:", "identity (unweighted Euclidean state norm)" if full_sample_weight_matrix is None else full_sample_weight_matrix.shape)
print("Quick test:", FULL_SAMPLE_QUICK_TEST, "maximum BCKE iterations:", FULL_SAMPLE_MAX_ITERATIONS)
display(pd.Series(asdict(full_sample_reconstruction_cfg), name="BCKE construction rules"))
display(pd.Series(asdict(full_sample_prediction_cfg), name="block KRR and decoder-aware correction"))

full_sample_results = run_bcke_experiment_full_sample(
    full_sample_fields_all, full_sample_mu_full, full_sample_weight_matrix, full_sample_reconstruction_cfg, full_sample_representation_grid, full_sample_prediction_cfg,
    n_train=FULL_SAMPLE_N_REPR_TRAIN, n_final=FULL_SAMPLE_N_FINAL_FIT, n_total=FULL_SAMPLE_N_TOTAL, split_mode=FULL_SAMPLE_SPLIT_MODE,
    refit=FULL_SAMPLE_REFIT_ON_TRAIN_VALIDATION, max_iterations=FULL_SAMPLE_MAX_ITERATIONS,
    audit=FULL_SAMPLE_CHECK_DIRECTIONAL_DERIVATIVES, audit_h=FULL_SAMPLE_DIRECTIONAL_H,
    audit_tolerance=FULL_SAMPLE_DIRECTIONAL_TOLERANCE, case_ids=FULL_SAMPLE_CASE_IDS,
)

# ============================================================================
# E. FINAL FORECAST: VARYING-M DMD a + LOCAL POLYNOMIAL z
# ============================================================================
# full_sample_results holds the kernel prediction baseline.
# full_sample_cell2_results holds the final DMD and local-polynomial forecast.
add_bcke_relative_errors(full_sample_results, full_sample_fields_all)
full_sample_cell2_begin = time.perf_counter()
full_sample_cell2_linear_predictor = LinearCoordinateVaryingM(
    full_sample_results["predictor"], window=FULL_SAMPLE_CELL2_DMD_WINDOW,
    ridge=FULL_SAMPLE_CELL2_DMD_RIDGE, variation_ridge=FULL_SAMPLE_CELL2_DMD_VARIATION_RIDGE,
    max_steps=FULL_SAMPLE_CELL2_DMD_MAX_STEPS,
)
full_sample_cell2_decoder_predictor = LocalLatentDecoderContinuation(
    full_sample_cell2_linear_predictor, window=FULL_SAMPLE_CELL2_DECODER_WINDOW,
    ridge=FULL_SAMPLE_CELL2_DECODER_RIDGE,
    max_horizon_steps=FULL_SAMPLE_CELL2_DECODER_MAX_HORIZON_STEPS,
)
full_sample_cell2_predictor = DirectPolynomialZContinuation(
    full_sample_cell2_decoder_predictor, mode="local",
    origins=FULL_SAMPLE_CELL2_POLY_CV_ORIGINS, horizon=FULL_SAMPLE_CELL2_POLY_CV_HORIZON,
    degrees=FULL_SAMPLE_CELL2_POLY_DEGREES, windows=FULL_SAMPLE_CELL2_POLY_WINDOWS,
    ridges=FULL_SAMPLE_CELL2_POLY_RIDGES, terminal_weight=FULL_SAMPLE_CELL2_POLY_TERMINAL_WEIGHT,
)
full_sample_cell2_setup_seconds = time.perf_counter()-full_sample_cell2_begin
full_sample_cell2_begin = time.perf_counter()
full_sample_cell2_prediction_all = full_sample_cell2_predictor.predict(full_sample_results["parameters"])
full_sample_cell2_prediction_seconds = time.perf_counter()-full_sample_cell2_begin

# Copy mutable reports so the later comparison cells retain their base results.
full_sample_cell2_results = dict(full_sample_results)
full_sample_cell2_results["predictor"] = full_sample_cell2_predictor
full_sample_cell2_results["prediction_all"] = full_sample_cell2_prediction_all
full_sample_cell2_results["per_case_errors"] = full_sample_results["per_case_errors"].copy()
full_sample_cell2_results["per_case_errors"]["prediction_weighted_error"] = np.sqrt(np.maximum(
    weighted_norm_sq_rows(full_sample_fields_all-full_sample_cell2_prediction_all, full_sample_results["problem"].W), 0.0))
add_bcke_relative_errors(full_sample_cell2_results, full_sample_fields_all)
full_sample_cell2_results["metrics"] = full_sample_results["metrics"].copy()
full_sample_cell2_before_refit = ((full_sample_cell2_results["metrics"]["stage"] != "final")
                      & (full_sample_cell2_results["metrics"]["mode"] == "parameter-only prediction"))
full_sample_cell2_results["metrics"].loc[full_sample_cell2_before_refit, "mode"] = "base KRR parameter-only prediction"
for full_sample_cell2_split, full_sample_cell2_ids in (("final fitting set", full_sample_results["final_fit_indices"]),
                              ("test (untouched)", full_sample_results["test_indices"])):
    full_sample_cell2_metric_mask = ((full_sample_cell2_results["metrics"]["stage"] == "final")
                         & (full_sample_cell2_results["metrics"]["split"] == full_sample_cell2_split)
                         & (full_sample_cell2_results["metrics"]["mode"] == "parameter-only prediction"))
    full_sample_cell2_metric = state_error(
        full_sample_fields_all[full_sample_cell2_ids], full_sample_cell2_prediction_all[full_sample_cell2_ids],
        full_sample_results["problem"].mean, full_sample_results["problem"].W)
    for full_sample_cell2_key, full_sample_cell2_value in full_sample_cell2_metric.items():
        full_sample_cell2_results["metrics"].loc[full_sample_cell2_metric_mask, full_sample_cell2_key] = full_sample_cell2_value
    full_sample_cell2_results["metrics"].loc[full_sample_cell2_metric_mask, "mode"] = "DMD a + local polynomial z"
full_sample_cell2_results["protocol"] = dict(full_sample_results["protocol"])
full_sample_cell2_results["protocol"]["final_parameter_predictor"] = full_sample_cell2_predictor.summary()
full_sample_cell2_results["timings"] = dict(full_sample_results["timings"])
full_sample_cell2_results["timings"]["base_prediction_all_seconds"] = full_sample_results["timings"]["prediction_all_seconds"]
full_sample_cell2_results["timings"]["local_polynomial_setup_seconds"] = full_sample_cell2_setup_seconds
full_sample_cell2_results["timings"]["prediction_all_seconds"] = full_sample_cell2_prediction_seconds
full_sample_cell2_results["timings"]["total_seconds"] += full_sample_cell2_setup_seconds+full_sample_cell2_prediction_seconds

display(full_sample_cell2_results["representation_candidates"])
display(full_sample_cell2_results["prediction_candidates"])
display(full_sample_cell2_results["derivative_audit"])
display(full_sample_cell2_results["prediction_derivative_audit"])
display(pd.Series(full_sample_cell2_predictor.summary(), name="final DMD + local polynomial predictor"))
display(full_sample_cell2_predictor.cv_results)
display(pd.Series(full_sample_cell2_results["problem"].summary(), name="final BCKE fit"))
display(pd.DataFrame(full_sample_cell2_results["problem"].bandwidth_report).T)
display(full_sample_cell2_results["metrics"])
display(full_sample_cell2_results["per_case_errors"].query("original_split == 'test'"))
display(pd.Series(full_sample_cell2_results["timings"], name="wall-clock times (seconds)"))

full_sample_output_directory = None
if FULL_SAMPLE_SAVE_OUTPUTS:
    # Reuse the original report writer for the representation and base map,
    # then add the complete active DMD/local-polynomial model components.
    full_sample_cell2_export_results = dict(full_sample_cell2_results)
    full_sample_cell2_export_results["predictor"] = full_sample_results["predictor"]
    full_sample_output_directory = save_bcke_results(full_sample_cell2_export_results, FULL_SAMPLE_OUTPUT_ROOT)
    (full_sample_output_directory/"parameter_model.npz").rename(full_sample_output_directory/"base_parameter_model.npz")
    with np.load(full_sample_output_directory/"base_parameter_model.npz", allow_pickle=False) as full_sample_cell2_archive:
        full_sample_cell2_model_arrays = {full_sample_key: full_sample_cell2_archive[full_sample_key].copy() for full_sample_key in full_sample_cell2_archive.files}
    full_sample_cell2_model_arrays.update(
        model_format=np.array("bcke_varying_m_local_poly_v1"),
        linear_G0=full_sample_cell2_linear_predictor.G0,
        linear_G1=full_sample_cell2_linear_predictor.G1,
        linear_c0=full_sample_cell2_linear_predictor.c0,
        linear_a_reference=full_sample_cell2_linear_predictor.a_reference,
        linear_anchor_shift=full_sample_cell2_linear_predictor.anchor_shift,
        linear_n_pairs=full_sample_cell2_linear_predictor.n_pairs,
        linear_max_steps=full_sample_cell2_linear_predictor.max_steps,
        mu_min=full_sample_cell2_linear_predictor.mu_min,
        mu_boundary=full_sample_cell2_linear_predictor.mu_boundary,
        grid_step=full_sample_cell2_linear_predictor.grid_step,
        decoder_J=full_sample_cell2_decoder_predictor.J,
        decoder_z_boundary=full_sample_cell2_decoder_predictor.z_boundary,
        decoder_residual_boundary=full_sample_cell2_decoder_predictor.residual_boundary,
        decoder_max_horizon_steps=full_sample_cell2_decoder_predictor.max_horizon_steps,
        polynomial_coefficients=full_sample_cell2_predictor.z_model.coefficients,
        polynomial_degree=full_sample_cell2_predictor.z_model.degree,
        polynomial_mu_scale=full_sample_cell2_predictor.z_model.mu_scale,
        polynomial_anchor_shift=full_sample_cell2_predictor.anchor_shift,
    )
    np.savez_compressed(full_sample_output_directory/"parameter_model.npz", **full_sample_cell2_model_arrays)
    full_sample_cell2_predictor.cv_results.to_csv(full_sample_output_directory/"local_polynomial_cv.csv", index=False)
    full_sample_cell2_metadata_path = full_sample_output_directory/"settings_and_summary.json"
    full_sample_cell2_metadata = json.loads(full_sample_cell2_metadata_path.read_text(encoding="utf-8"))
    full_sample_cell2_metadata["base_predictor"] = full_sample_cell2_metadata["selected_predictor"]
    full_sample_cell2_metadata["selected_predictor"] = full_sample_cell2_predictor.summary()
    full_sample_cell2_metadata["linear_extrapolator"] = full_sample_cell2_linear_predictor.summary()
    full_sample_cell2_metadata["nonlinear_decoder"] = full_sample_cell2_decoder_predictor.summary()
    full_sample_cell2_metadata["model_format"] = "bcke_varying_m_local_poly_v1"
    full_sample_cell2_metadata["note"] = (
        "Cell 2 final prediction uses varying-M DMD a and direct local-polynomial z. "
        "The fitted interval retains the base map. The original representation and "
        "base KRR selection tables are retained. Local-polynomial CV freezes the "
        "representation and uses only known fitting coordinates. "
        "The test block has previously been viewed; comparisons are exploratory.")
    full_sample_cell2_metadata_path.write_text(json.dumps(full_sample_cell2_metadata, indent=2, ensure_ascii=False),
                                  encoding="utf-8")

# Plot fitting diagnostics, reconstruction errors, and prediction errors.
plot_bcke_results(full_sample_cell2_results, full_sample_output_directory, FULL_SAMPLE_FIGURE_DPI)
plot_bcke_parameter_errors(
    full_sample_cell2_results, full_sample_output_directory, FULL_SAMPLE_FIGURE_DPI,
    parameter_axis=FULL_SAMPLE_PARAMETER_AXIS, parameter_label=FULL_SAMPLE_PARAMETER_AXIS_LABEL,
)
if full_sample_output_directory is not None:
    print("Saved to:", full_sample_output_directory.resolve())
print("Final optimizer status:", full_sample_cell2_results["problem"].status)
print("Cell 2 uses reconstruction on all fitting snapshots.")
print("Forecast: varying-M DMD linear a + direct local-polynomial nonlinear z.")

# Optional parameter-only use after fitting (run only Cells 1 and 2):
# mu_query = np.array([[your_parameter_value]])
# u_query = full_sample_cell2_predictor.predict(mu_query)  # shape (n_queries, n_dofs)
# full_sample_cell2_results["predictor"] is the same active predictor.
# parameter_model.npz stores its complete components in the new format.
# The legacy predict_saved_bcke loader supports only base_parameter_model.npz.


# ============================================================================
# F. SAME-DATA COMPARISON: BCKE, POD+RBF, AND PPMD FROM SOURCE CELL 4
# ============================================================================
# PPMD follows the executed source Cell 4: fit PCA and diffusion on the
# fitting prefix; forecast linear coordinates recursively; predict future
# high-dimensional residuals directly from the parameter.
FULL_SAMPLE_COMPARISON_POD_ALPHAS = (1e-8, 1e-6, 1e-4, 1e-2, 1.0)
FULL_SAMPLE_COMPARISON_POD_GAMMAS = (1e-2, 1e-1, 1.0, 10.0, 100.0)
FULL_SAMPLE_COMPARISON_POD_CV_FOLDS = 5
# PPMD reconstruction and parameter prediction both use source Cell 4
# (8 linear coordinates, 4 diffusion coordinates); each fit uses fitting states only.
FULL_SAMPLE_COMPARISON_PPMD_RECONSTRUCTION_LINEAR_DIM = 2 if FULL_SAMPLE_DATA_SOURCE == "synthetic" else 8
FULL_SAMPLE_COMPARISON_PPMD_RECONSTRUCTION_NONLINEAR_DIM = 2 if FULL_SAMPLE_DATA_SOURCE == "synthetic" else 4
FULL_SAMPLE_COMPARISON_PPMD_RECONSTRUCTION_CONFIG = dict(
    residual_neighbors=15,
    diffusion_k=22, diffusion_epsilon=100.0, diffusion_alpha=1.6,
    lifting_grid={
        "poly_features__degree": [2],
        "krr__alpha": [1e-4],
        "krr__degree": [3],
        "krr__coef0": [10],
    },
    lifting_cv_folds=10, n_jobs=-1,
)
if FULL_SAMPLE_DATA_SOURCE == "synthetic":
    FULL_SAMPLE_COMPARISON_PPMD_RECONSTRUCTION_CONFIG = dict(
        FULL_SAMPLE_COMPARISON_PPMD_RECONSTRUCTION_CONFIG, lifting_cv_folds=5, n_jobs=1)

# Source PPMD Cell 4: 8 linear coordinates and 4 diffusion coordinates.
# The explicit synthetic smoke test retains its smaller 2+2 dimensions.
FULL_SAMPLE_COMPARISON_PPMD_LINEAR_DIM = 2 if FULL_SAMPLE_DATA_SOURCE == "synthetic" else 8
FULL_SAMPLE_COMPARISON_PPMD_NONLINEAR_DIM = 2 if FULL_SAMPLE_DATA_SOURCE == "synthetic" else 4
FULL_SAMPLE_COMPARISON_PPMD_CONFIG = dict(
    residual_neighbors=15,
    diffusion_k=22, diffusion_epsilon=100.0, diffusion_alpha=1.6,
    lifting_grid={
        "poly_features__degree": [2],
        "krr__alpha": [1e-4],
        "krr__degree": [3],
        "krr__coef0": [10],
    },
    lifting_cv_folds=10,
    # (last forecast step, polynomial degree, alpha, coef0); linear branch only.
    linear_schedule=(
        (3, 4, 4e-1, 55), (5, 3, 1e-2, 3), (7, 3, 8e-2, 10),
        (8, 3, 8e-2, 32000), (9, 4, 7, 22), (10, 3, 1e-6, 10),
    ),
    nonlinear_grid={
        "trend_degree": [3], "trend_alpha": [300],
        "residual_alpha": [1e-5], "residual_gamma": [7.8],
    },
    nonlinear_cv_origins=(50, 60, 70, 75, 80),
    nonlinear_cv_horizon=10,
    n_jobs=-1,
)
FULL_SAMPLE_COMPARISON_PPMD_QUERY_PARAMETERS = (
    2.55e-4, 5.05e-4, 7.55e-4, 9.25e-4,
    9.45e-4, 9.65e-4, 9.85e-4, 10.05e-4,
)
if FULL_SAMPLE_DATA_SOURCE == "synthetic":
    # This 24-case check uses fewer neighbors; physical-data defaults above
    # are the exact values from the supplied 100-case script.
    FULL_SAMPLE_COMPARISON_PPMD_CONFIG = dict(FULL_SAMPLE_COMPARISON_PPMD_CONFIG,
        diffusion_k=min(22, FULL_SAMPLE_N_FINAL_FIT-1), n_jobs=1,
        nonlinear_cv_origins=(10, 12, 16), nonlinear_cv_horizon=4)
    FULL_SAMPLE_COMPARISON_PPMD_QUERY_PARAMETERS = ()

full_sample_comparison_mu, full_sample_comparison_fit, full_sample_comparison_test = _validate_ordered_scalar_comparison(
    full_sample_results["parameters"], full_sample_results["final_fit_indices"], full_sample_results["test_indices"])
full_sample_comparison_total_dimension = full_sample_cell2_results["problem"].p+full_sample_cell2_results["problem"].d
if FULL_SAMPLE_COMPARISON_PPMD_LINEAR_DIM+FULL_SAMPLE_COMPARISON_PPMD_NONLINEAR_DIM != full_sample_comparison_total_dimension:
    raise ValueError("Set the two PPMD dimensions to the BCKE total dimension for a fair comparison.")
if FULL_SAMPLE_COMPARISON_PPMD_RECONSTRUCTION_LINEAR_DIM+FULL_SAMPLE_COMPARISON_PPMD_RECONSTRUCTION_NONLINEAR_DIM != full_sample_comparison_total_dimension:
    raise ValueError("PPMD reconstruction dimensions must also match the BCKE total dimension.")

full_sample_pod_rbf_prediction_all, full_sample_pod_rbf_reference_info = fit_pod_rbf_reference(
    full_sample_fields_all, full_sample_comparison_mu, full_sample_comparison_fit, full_sample_comparison_total_dimension,
    full_sample_cell2_results["problem"].W, alphas=FULL_SAMPLE_COMPARISON_POD_ALPHAS,
    gammas=FULL_SAMPLE_COMPARISON_POD_GAMMAS, cv_folds=FULL_SAMPLE_COMPARISON_POD_CV_FOLDS,
    seed=full_sample_reconstruction_cfg.seed,
)
full_sample_pod_rbf_reconstruction_fit = full_sample_pod_rbf_reference_info.pop("fitting_reconstruction")
full_sample_ppmd_prediction_all, full_sample_ppmd_reference_info, full_sample_ppmd_query_model = fit_ppmd_reference(
    full_sample_fields_all, full_sample_comparison_mu, full_sample_comparison_fit,
    linear_dimension=FULL_SAMPLE_COMPARISON_PPMD_LINEAR_DIM,
    nonlinear_dimension=FULL_SAMPLE_COMPARISON_PPMD_NONLINEAR_DIM,
    seed=full_sample_reconstruction_cfg.seed, return_model=True,
    **FULL_SAMPLE_COMPARISON_PPMD_CONFIG,
)
# Queries use linear PCHIP; fitting-interval residuals use diffusion lifting,
# and future residuals use the direct parameter model from source Cell 4.
if FULL_SAMPLE_COMPARISON_PPMD_QUERY_PARAMETERS:
    full_sample_ppmd_query_parameters = parameter_array(FULL_SAMPLE_COMPARISON_PPMD_QUERY_PARAMETERS)
    full_sample_ppmd_query_predictions = predict_ppmd_cell4(
        full_sample_ppmd_query_parameters, full_sample_ppmd_query_model)
else:
    full_sample_ppmd_query_parameters = np.empty((0, 1), dtype=np.float64)
    full_sample_ppmd_query_predictions = np.empty((0, full_sample_fields_all.shape[1]), dtype=np.float64)
full_sample_ppmd_reconstruction_fit, full_sample_ppmd_reconstruction_info = fit_ppmd_previous_reconstruction_reference(
    full_sample_fields_all, full_sample_comparison_mu, full_sample_comparison_fit,
    linear_dimension=FULL_SAMPLE_COMPARISON_PPMD_RECONSTRUCTION_LINEAR_DIM,
    nonlinear_dimension=FULL_SAMPLE_COMPARISON_PPMD_RECONSTRUCTION_NONLINEAR_DIM,
    seed=full_sample_reconstruction_cfg.seed,
    **FULL_SAMPLE_COMPARISON_PPMD_RECONSTRUCTION_CONFIG,
)
full_sample_ppmd_reference_info["reconstruction_reference"] = full_sample_ppmd_reconstruction_info
full_sample_ppmd_reference_info["separate_reconstruction_and_prediction_settings"] = True
full_sample_comparison_predictions = {
    "BCKE": full_sample_cell2_prediction_all,
    "POD+RBF": full_sample_pod_rbf_prediction_all,
    "PPMD": full_sample_ppmd_prediction_all,
}
# Retain the two error regions: known-state reconstruction on the
# fitting prefix and parameter-only prediction on the final test block.
full_sample_comparison_estimates = {full_sample_name: full_sample_estimate.copy()
                        for full_sample_name, full_sample_estimate in full_sample_comparison_predictions.items()}
full_sample_comparison_estimates["BCKE"][full_sample_comparison_fit] = full_sample_cell2_results["reconstruction_all"][full_sample_comparison_fit]
full_sample_comparison_estimates["POD+RBF"][full_sample_comparison_fit] = full_sample_pod_rbf_reconstruction_fit
# Use the previous PPMD reconstruction only on fitting rows.
# Test rows retain the source-Cell-4 parameter prediction.
full_sample_comparison_estimates["PPMD"][full_sample_comparison_fit] = full_sample_ppmd_reconstruction_fit
full_sample_comparison_errors = build_bcke_comparison_table(
    full_sample_fields_all, full_sample_comparison_mu, full_sample_comparison_estimates,
    full_sample_comparison_fit, full_sample_comparison_test, full_sample_cell2_results["problem"].W,
)
full_sample_comparison_errors["evaluation"] = np.where(
    full_sample_comparison_errors.region == "fitting", "training reconstruction", "extrapolation prediction")
full_sample_comparison_test_errors = full_sample_comparison_errors.loc[
    full_sample_comparison_errors.region == "test"
].pivot(index="snapshot_index", columns="method", values="relative_weighted_L2")
full_sample_comparison_summary = full_sample_comparison_errors.groupby(["method", "region"])[
    ["relative_weighted_L2", "absolute_weighted_L2"]
].agg(["mean", "max"])

display(pd.Series(full_sample_pod_rbf_reference_info, name="POD+RBF reference"))
display(pd.Series(full_sample_ppmd_reconstruction_info, name="PPMD previous reconstruction reference"))
display(pd.Series({full_sample_key: full_sample_value for full_sample_key, full_sample_value in full_sample_ppmd_reference_info.items()
                   if full_sample_key != "nonlinear_cv_results"}, name="PPMD source Cell 4 prediction"))
display(pd.DataFrame(full_sample_ppmd_reference_info["nonlinear_cv_results"]))
display(full_sample_comparison_summary)
display(full_sample_comparison_test_errors)
plot_bcke_method_comparison(
    full_sample_comparison_errors, full_sample_comparison_fit, full_sample_comparison_test,
    parameter_label=FULL_SAMPLE_PARAMETER_AXIS_LABEL, output_directory=full_sample_output_directory,
    dpi=FULL_SAMPLE_FIGURE_DPI,
)
if full_sample_output_directory is not None:
    full_sample_comparison_errors.to_csv(full_sample_output_directory/"method_comparison_per_case.csv", index=False)
    full_sample_comparison_summary.to_csv(full_sample_output_directory/"method_comparison_summary.csv")
    pd.DataFrame(full_sample_ppmd_reference_info["nonlinear_cv_results"]).to_csv(
        full_sample_output_directory/"ppmd_direct_highdim_cv.csv", index=False)
    with (full_sample_output_directory/"comparison_reference_settings.json").open("w", encoding="utf-8") as full_sample_comparison_file:
        json.dump(dict(pod_rbf=full_sample_pod_rbf_reference_info, ppmd=full_sample_ppmd_reference_info),
                  full_sample_comparison_file, indent=2, ensure_ascii=False)
    np.savez_compressed(
        full_sample_output_directory/"method_comparison_predictions.npz",
        bcke=full_sample_cell2_prediction_all, pod_rbf=full_sample_pod_rbf_prediction_all,
        ppmd=full_sample_ppmd_prediction_all, parameters=full_sample_comparison_mu,
        ppmd_reconstruction_fit=full_sample_ppmd_reconstruction_fit,
        ppmd_query_parameters=full_sample_ppmd_query_parameters,
        ppmd_query_predictions=full_sample_ppmd_query_predictions,
        bcke_comparison_estimates=full_sample_comparison_estimates["BCKE"],
        pod_rbf_comparison_estimates=full_sample_comparison_estimates["POD+RBF"],
        ppmd_comparison_estimates=full_sample_comparison_estimates["PPMD"],
        final_fit_indices=full_sample_comparison_fit, test_indices=full_sample_comparison_test,
    )
print("PPMD uses source Cell 4: 8+4 dimensions, linear recursive KRR + PCHIP, direct nonlinear residual prediction.")
print("Comparison figures: training reconstruction and extrapolation prediction, both on log error axes.")
print("Comparison rows and error norms are shared; PPMD preprocessing uses fitting states only.")
print("The final ten test states were viewed in earlier experiments; treat this comparison as exploratory.")


In [ ]:
# Cell 3: cylinder fields and VTUs at the paper comparison parameters.
# Run Cells 1 and 2. Rerun this cell to change the field queries.
import meshio

if "full_sample_cell2_results" not in globals():
    raise RuntimeError("Run the cylinder experiment in Cell 2 first.")

CYLINDER_FLOW_DATA_ROOT = CYLINDER_VTU_ROOT
CYLINDER_FLOW_VTU_FILENAME = "circle-2d-drag_150.vtu"
CYLINDER_FLOW_COMPONENT = FULL_SAMPLE_STATE_COMPONENT
CYLINDER_FLOW_RECONSTRUCTION_INDICES = (38,)  # mu=4.0e-4
CYLINDER_FLOW_QUERY_PARAMETERS = (2.55e-4, 9.65e-4)
CYLINDER_FLOW_EXPORT_ALL_TEST = False
CYLINDER_FLOW_SAVE_VTU = True
CYLINDER_FLOW_FIGURE_DPI = FULL_SAMPLE_FIGURE_DPI
CYLINDER_FLOW_OUTPUT_ROOT = (OUTPUT_DIR / "bcke_cylinder_flow_vtu_outputs")

# Evaluate the already fitted parameter predictor before reading query references.
cylinder_flow_query_mu = parameter_array(np.asarray(CYLINDER_FLOW_QUERY_PARAMETERS, dtype=np.float64)[:, None])
cylinder_flow_query_predictions = full_sample_cell2_results["predictor"].predict(cylinder_flow_query_mu)
cylinder_flow_query_paths = [
    resolve_bcke_case_vtu(
        CYLINDER_FLOW_DATA_ROOT, cylinder_flow_parameter, CYLINDER_FLOW_VTU_FILENAME,
        folder_prefix="flow", parameter_scale=1e-4)
    for cylinder_flow_parameter in cylinder_flow_query_mu[:, 0]
]
cylinder_flow_query_reference = np.vstack([
    _bcke_vtu_state(meshio.read(cylinder_flow_path), CYLINDER_FLOW_COMPONENT)
    for cylinder_flow_path in cylinder_flow_query_paths
])
if (cylinder_flow_query_predictions.shape != cylinder_flow_query_reference.shape
        or not np.isfinite(cylinder_flow_query_predictions).all()):
    raise ValueError("Query predictions must match the reference VTU state size and be finite.")

# These appended rows are only a display view. The original fitting data,
# representation, predictor, and result dictionary are retained as supplied.
cylinder_flow_original_states = np.asarray(full_sample_fields_all, dtype=np.float64)
cylinder_flow_original_mu = parameter_array(full_sample_cell2_results["parameters"])
if not np.allclose(
        cylinder_flow_original_mu[np.asarray(CYLINDER_FLOW_RECONSTRUCTION_INDICES), 0],
        4.0e-4, rtol=1e-12, atol=1e-15):
    raise ValueError("The reconstruction row must represent the paper viscosity 4.0e-4.")
cylinder_flow_query_indices = np.arange(
    len(cylinder_flow_original_states),
    len(cylinder_flow_original_states)+len(cylinder_flow_query_mu), dtype=int)
cylinder_flow_display_states = np.vstack((cylinder_flow_original_states, cylinder_flow_query_reference))
cylinder_flow_display_results = dict(full_sample_cell2_results)
cylinder_flow_display_results["parameters"] = np.vstack((cylinder_flow_original_mu, cylinder_flow_query_mu))
cylinder_flow_display_results["test_indices"] = cylinder_flow_query_indices
cylinder_flow_display_results["prediction_all"] = np.vstack((
    full_sample_cell2_results["prediction_all"], cylinder_flow_query_predictions))
cylinder_flow_display_results["reconstruction_all"] = np.vstack((
    full_sample_cell2_results["reconstruction_all"], np.full_like(cylinder_flow_query_reference, np.nan)))

# Use a separate output directory for this paper selection.
cylinder_flow_export = plot_and_export_bcke_case_flow(
    cylinder_flow_display_results, cylinder_flow_display_states,
    data_root=CYLINDER_FLOW_DATA_ROOT, filename=CYLINDER_FLOW_VTU_FILENAME,
    folder_prefix="flow", parameter_scale=1e-4, component=CYLINDER_FLOW_COMPONENT,
    reconstruction_indices=CYLINDER_FLOW_RECONSTRUCTION_INDICES,
    selected_test_indices=cylinder_flow_query_indices, query_parameters=(),
    export_all_test=CYLINDER_FLOW_EXPORT_ALL_TEST, save_vtu=CYLINDER_FLOW_SAVE_VTU,
    output_directory=Path(full_sample_output_directory)/"paper_parameters",
    output_root=CYLINDER_FLOW_OUTPUT_ROOT, dpi=CYLINDER_FLOW_FIGURE_DPI,
)

# Mark independent query references explicitly in the VTU manifest.
cylinder_flow_manifest = cylinder_flow_export["vtu_manifest"].copy()
if len(cylinder_flow_manifest):
    cylinder_flow_manifest["query_index"] = -1
    cylinder_flow_manifest["parameter_region"] = "fitting reconstruction"
    cylinder_flow_fitting_boundary = float(np.max(
        cylinder_flow_original_mu[full_sample_cell2_results["final_fit_indices"], 0]))
    for cylinder_flow_query_number, cylinder_flow_display_index in enumerate(cylinder_flow_query_indices):
        cylinder_flow_mask = ((cylinder_flow_manifest["mode"] == "prediction")
                             & (cylinder_flow_manifest["snapshot_index"] == cylinder_flow_display_index))
        cylinder_flow_manifest.loc[cylinder_flow_mask, "snapshot_index"] = -1
        cylinder_flow_manifest.loc[cylinder_flow_mask, "query_index"] = cylinder_flow_query_number
        cylinder_flow_manifest.loc[cylinder_flow_mask, "parameter_region"] = (
            "interpolation" if cylinder_flow_query_mu[cylinder_flow_query_number, 0] <= cylinder_flow_fitting_boundary
            else "extrapolation")
    cylinder_flow_manifest.to_csv(cylinder_flow_export["output_directory"]/"vtu_manifest.csv", index=False)
    cylinder_flow_export["vtu_manifest"] = cylinder_flow_manifest
    display(cylinder_flow_manifest[["mode", "parameter", "parameter_region", "relative_L2",
                                   "field_vtu", "error_vtu"]])
cylinder_flow_export["prediction_query_parameters"] = cylinder_flow_query_mu[:, 0].copy()
cylinder_flow_export["query_reference_vtus"] = tuple(map(str, cylinder_flow_query_paths))

# All-method fields use the same comparison fits and exact paper queries.
if "full_sample_comparison_estimates" not in globals():
    raise RuntimeError("Run the complete cylinder experiment, including its comparison section, first.")
cylinder_flow_pod_rbf_queries = query_pod_rbf_flow(
    full_sample_fields_all, full_sample_comparison_mu, full_sample_comparison_fit,
    full_sample_comparison_total_dimension, full_sample_cell2_results["problem"].W,
    full_sample_pod_rbf_reference_info, cylinder_flow_query_mu,
    grid_prediction=full_sample_pod_rbf_prediction_all,
)
cylinder_flow_ppmd_queries = predict_ppmd_cell4(
    cylinder_flow_query_mu, full_sample_ppmd_query_model)
cylinder_flow_rec_index = int(CYLINDER_FLOW_RECONSTRUCTION_INDICES[0])
cylinder_flow_rec_parameter = float(cylinder_flow_original_mu[cylinder_flow_rec_index, 0])
cylinder_flow_reconstruction_vtu = resolve_bcke_case_vtu(
    CYLINDER_FLOW_DATA_ROOT, cylinder_flow_rec_parameter, CYLINDER_FLOW_VTU_FILENAME,
    folder_prefix="flow", parameter_scale=1e-4)
cylinder_flow_method_comparison = plot_and_export_flow_method_comparison(
    full_sample_fields_all[cylinder_flow_rec_index],
    {name: estimates[cylinder_flow_rec_index] for name, estimates in full_sample_comparison_estimates.items()},
    cylinder_flow_query_reference,
    {"BCKE": cylinder_flow_query_predictions, "POD+RBF": cylinder_flow_pod_rbf_queries, "PPMD": cylinder_flow_ppmd_queries},
    reconstruction_parameter=cylinder_flow_rec_parameter, query_parameters=cylinder_flow_query_mu,
    reconstruction_vtu=cylinder_flow_reconstruction_vtu, query_vtus=cylinder_flow_query_paths,
    component=CYLINDER_FLOW_COMPONENT, case_name="cylinder",
    dimension=full_sample_comparison_total_dimension,
    output_directory=cylinder_flow_export["output_directory"],
    dpi=CYLINDER_FLOW_FIGURE_DPI, save_vtu=CYLINDER_FLOW_SAVE_VTU,
)


In [ ]:
# Cell 4: independent cylinder prediction debugging with trend + kernel z.
# Run Cells 1 and 2 first. Change settings here and rerun only this cell.
# The fitting representation, linear forecast, and local lifting are reused.

CYLINDER_TREND_KERNEL_CV_ORIGINS = (50,)
CYLINDER_TREND_KERNEL_CV_HORIZON = 20
CYLINDER_TREND_KERNEL_DEGREES = (1,)
CYLINDER_TREND_KERNEL_WINDOWS = (None,)  # None uses a global polynomial trend
CYLINDER_TREND_KERNEL_LENGTHS = (0.17,)  # normalized fitting interval
CYLINDER_TREND_KERNEL_RIDGES = (1e-7,)
CYLINDER_TREND_KERNEL_TREND_RIDGE = 1e-10
CYLINDER_TREND_KERNEL_TERMINAL_WEIGHT = 0.0  # mean CV error; no extra terminal weight
CYLINDER_TREND_KERNEL_SAVE_OUTPUTS = True
CYLINDER_TREND_KERNEL_OUTPUT_ROOT = (OUTPUT_DIR / "bcke_cylinder_trend_kernel_z_outputs")
CYLINDER_TREND_KERNEL_FIGURE_DPI = 220

if "run_bcke_trend_kernel_prediction_debug" not in globals():
    raise RuntimeError("Run Cell 1 with the prediction debugging functions first.")
if "full_sample_cell2_results" not in globals() or "full_sample_cell2_decoder_predictor" not in globals():
    raise RuntimeError("Run the cylinder experiment in Cell 2 first.")

cylinder_trend_kernel_debug = run_bcke_trend_kernel_prediction_debug(
    full_sample_cell2_results, full_sample_fields_all,
    full_sample_cell2_decoder_predictor, case_name="Cylinder",
    origins=CYLINDER_TREND_KERNEL_CV_ORIGINS,
    horizon=CYLINDER_TREND_KERNEL_CV_HORIZON,
    degrees=CYLINDER_TREND_KERNEL_DEGREES,
    trend_windows=CYLINDER_TREND_KERNEL_WINDOWS,
    length_scales=CYLINDER_TREND_KERNEL_LENGTHS,
    kernel_ridges=CYLINDER_TREND_KERNEL_RIDGES,
    trend_ridge=CYLINDER_TREND_KERNEL_TREND_RIDGE,
    terminal_weight=CYLINDER_TREND_KERNEL_TERMINAL_WEIGHT,
    save_outputs=CYLINDER_TREND_KERNEL_SAVE_OUTPUTS,
    output_root=CYLINDER_TREND_KERNEL_OUTPUT_ROOT,
    dpi=CYLINDER_TREND_KERNEL_FIGURE_DPI,
)
cylinder_trend_kernel_predictor = cylinder_trend_kernel_debug["predictor"]

# Query only the parameter after fitting:
# u_query = cylinder_trend_kernel_predictor.predict([[your_parameter_value]])
# z_query = cylinder_trend_kernel_predictor.nonlinear_coordinates([[your_parameter_value]])


In [ ]:
# Cell 5: backward-facing step BCKE experiment with full-sample reconstruction.
# Run Cell 1 before this cell. Results use the step_full_sample_ prefix.
# Run Cell 1 once; after changing settings, rerun only this cell.
# PyTorch supplies matrix operations and automatic differentiation, NOT a neural network.

# ============================================================================
# A. DATA AND EXPERIMENT PROTOCOL
# ============================================================================
STEP_FULL_SAMPLE_DATA_SOURCE = "npz"                # "ppmd", "npz", or "synthetic"
STEP_FULL_SAMPLE_PPMD_MODULE_DIRECTORY = None  # folder containing Diffusion.py
STEP_FULL_SAMPLE_DATA_ROOT = STEP_VTU_ROOT
STEP_FULL_SAMPLE_VTU_FILENAME = "backward_facing_step_2d_25.vtu"
STEP_FULL_SAMPLE_STATE_COMPONENT = "u"              # same u-only state as the previous notebook; "uv" uses both
STEP_FULL_SAMPLE_NPZ_DATA_PATH = (DATA_DIR / "step.npz")  # keys: states (N,Nh), parameters (N,q); optional weights, case_ids
STEP_FULL_SAMPLE_WEIGHTS_PATH = None                # .npy diagonal mass weights or full SPD W; None uses identity
STEP_FULL_SAMPLE_PARAMETER_FILE = None              # optional .npy array of TRUE input parameters in snapshot order

# Step viscosity grid matches step2e-5, step4e-5, ..., step200e-5.
# Replace this with the actual labels used in your simulation, or set PARAMETER_FILE.
# For a genuinely regenerated Re=50,100,...,5000 dataset in that order, use:
# PARAMETER_VALUES = np.arange(50.0, 5000.0 + 50.0, 50.0)
STEP_FULL_SAMPLE_PARAMETER_VALUES = np.arange(1, 101, dtype=np.float64)*2e-5
STEP_FULL_SAMPLE_CASE_IDS = None                    # optional physical-case labels; all times of a case stay together

STEP_FULL_SAMPLE_N_TOTAL = 100                      # complete parameter cases, not spatial degrees of freedom
STEP_FULL_SAMPLE_N_REPR_TRAIN = 80
STEP_FULL_SAMPLE_N_FINAL_FIT = 90                   # 80 train + 10 validation + 10 untouched test
STEP_FULL_SAMPLE_SPLIT_MODE = "ordered"             # file order; "random" gives seeded case-wise interpolation split
STEP_FULL_SAMPLE_REFIT_ON_TRAIN_VALIDATION = True

STEP_FULL_SAMPLE_QUICK_TEST = False                  # True is a pipeline check, NOT a converged publication run
STEP_FULL_SAMPLE_QUICK_ITERATIONS = 30
STEP_FULL_SAMPLE_FULL_ITERATIONS = 600
STEP_FULL_SAMPLE_TORCH_THREADS = 1                  # small dense sample-space systems often benefit from one thread

# ============================================================================
# B. RECONSTRUCTION: CORE BCKE PARAMETERS (notation follows the LaTeX)
# ============================================================================
step_full_sample_reconstruction_cfg = BCKEFullSampleConfig(
    p=7, d=5, max_total_dimension=12,
    k_start=10,
    virtual_fractions=(0.2, 0.4, 0.6, 0.8),
    lambda_phi=5e-3,                # coordinate-map ridge, lambda_Phi
    lambda_lambda=8e-4,             # residual-map ridge, lambda_Lambda
    sigma_phi_factor=1,           # times the fixed residual distance quantile
    sigma_lambda_factor=0.3,        # times the initial coordinate distance quantile
    eta_a=1.0, alpha_x=1.0, alpha_z=1.0, beta_f=1.0, beta_b=1.0,
    omega_delta=0.01,
    gamma_diff=0.0,                 # 0: core BCKE; >0: the separate BCKE-D extension
    allow_ablation=True,           # only enable for explicitly labelled zero-weight comparisons
    distance_quantile=0.7,
    affinity_factors=(1.0, 1.0, 1.0, 1.0),  # epsilon_X, epsilon_hatZ, epsilon_V, epsilon_LambdaV
    bandwidth_fallback=0.5,
    eps_scale=1e-12,
    degree_safe=1e-14,
    bandwidth_degree_floor=1e-8,
    bandwidth_growth=1.2,
    max_bandwidth_expansions=200,
    rank_eigenvalue_rtol=1e-12,
    residual_zero_rtol=0.0,
    armijo_c=1e-4, armijo_rho=0.9, initial_step=1.0,
    grad_tol=1e-7, max_backtracks=1000,
    verbose_every=100,
    diff_chunk_size=64,
    seed=42, device="cpu",          # float64 throughout; CUDA requires a matching PyTorch installation
)

# One candidate per quantity is the default for a manageable first run.
# Expand these lists for the paper's finite-grid validation selection.
# This grid OVERRIDES the corresponding six values in reconstruction_cfg.
step_full_sample_representation_grid = dict(
    p=(step_full_sample_reconstruction_cfg.p,),
    d=(step_full_sample_reconstruction_cfg.d,),
    lambda_phi=(step_full_sample_reconstruction_cfg.lambda_phi,),
    lambda_lambda=(step_full_sample_reconstruction_cfg.lambda_lambda,),
    sigma_phi_factor=(step_full_sample_reconstruction_cfg.sigma_phi_factor,),
    sigma_lambda_factor=(step_full_sample_reconstruction_cfg.sigma_lambda_factor,),
)

# ============================================================================
# C. PREDICTION: BLOCK PARAMETER KRR + DECODER-AWARE RESIDUAL CORRECTION
# ============================================================================
step_full_sample_prediction_cfg = PredictionConfig(
    # The old shared Gaussian grid below remains an exact, zero-mean baseline.
    # Each pair is (linear a block, nonlinear z block); a is ignored when p=0.
    block_pairs=(
        (KernelBlockConfig("gaussian", 1.0, 1e-4), KernelBlockConfig("gaussian", 0.5, 1e-2)),
        (KernelBlockConfig("gaussian", 2.0, 1e-4), KernelBlockConfig("matern52", 1.0, 1e-4)),
        (KernelBlockConfig("matern52", 1.0, 1e-4), KernelBlockConfig("matern52", 0.3, 1e-4)),
    ),
    # For multi-parameter anisotropy, add fixed pairs with axis_factors=(...).
    # Values are positive multipliers, one per input column; choose before validation.
    correction_nus=(0.1, 0.5),  # B=0 is always included separately
    correction_max_iterations=30 if STEP_FULL_SAMPLE_QUICK_TEST else 200,
    correction_grad_tol=1e-7,
    correction_initial_step=1.0,
    correction_armijo_c=1e-4, correction_armijo_rho=0.8,
    correction_max_backtracks=40, correction_verbose_every=0,
    sigma_factors=(0.5, 1.0, ),
    ridges=(1e-8, 1e-6, ),  # one value also works: ridges=1e-4 or (1e-4,)
    distance_quantile=0.6,
    bandwidth_fallback=1.0,
)
# Every candidate uses [A; hat Z]. Separate a/z kernels are allowed.
# Only B is optimized in the correction; the BCKE decoder stays frozen.
# Joint selection uses full-state validation prediction error.

# Forecast settings for the backward-facing step experiment.
# All definitions are already in Cell 1; no later cell is required.
STEP_FULL_SAMPLE_CELL2_DMD_WINDOW = 60
STEP_FULL_SAMPLE_CELL2_DMD_RIDGE = 1e-07
STEP_FULL_SAMPLE_CELL2_DMD_VARIATION_RIDGE = 0.01
STEP_FULL_SAMPLE_CELL2_DMD_MAX_STEPS = 2000
STEP_FULL_SAMPLE_CELL2_DECODER_WINDOW = 3
STEP_FULL_SAMPLE_CELL2_DECODER_RIDGE = 1e-04
STEP_FULL_SAMPLE_CELL2_DECODER_MAX_HORIZON_STEPS = 10
STEP_FULL_SAMPLE_CELL2_POLY_CV_ORIGINS = (70,)
STEP_FULL_SAMPLE_CELL2_POLY_CV_HORIZON = 10
STEP_FULL_SAMPLE_CELL2_POLY_DEGREES = (3,)
STEP_FULL_SAMPLE_CELL2_POLY_WINDOWS = (15,)
STEP_FULL_SAMPLE_CELL2_POLY_RIDGES = (4e-04,)
STEP_FULL_SAMPLE_CELL2_POLY_TERMINAL_WEIGHT = 0.5

STEP_FULL_SAMPLE_CHECK_DIRECTIONAL_DERIVATIVES = True
STEP_FULL_SAMPLE_DIRECTIONAL_H = 1e-6
STEP_FULL_SAMPLE_DIRECTIONAL_TOLERANCE = 5e-4
STEP_FULL_SAMPLE_SAVE_OUTPUTS = True
STEP_FULL_SAMPLE_OUTPUT_ROOT = (OUTPUT_DIR / "bcke_step_full_sample_outputs")
STEP_FULL_SAMPLE_FIGURE_DPI = 220

# Only the NEW figures use these axis settings; they do not affect fitting.
STEP_FULL_SAMPLE_PARAMETER_AXIS = 0  # zero-based input column; a scalar parameter uses 0
STEP_FULL_SAMPLE_PARAMETER_AXIS_LABEL = r"parameter $\mu$"  # for Reynolds data: "Reynolds number Re"

# ============================================================================
# D. LOAD DATA AND RUN THE COMPLETE PROTOCOL
# ============================================================================
torch.set_num_threads(STEP_FULL_SAMPLE_TORCH_THREADS)
torch.manual_seed(step_full_sample_reconstruction_cfg.seed)
np.random.seed(step_full_sample_reconstruction_cfg.seed)
if STEP_FULL_SAMPLE_DATA_SOURCE == "ppmd":
    step_full_sample_fields_all = load_ppmd_step_states(
        STEP_FULL_SAMPLE_STATE_COMPONENT, STEP_FULL_SAMPLE_PPMD_MODULE_DIRECTORY,
        data_root=STEP_FULL_SAMPLE_DATA_ROOT, filename=STEP_FULL_SAMPLE_VTU_FILENAME,
        parameters=(np.load(STEP_FULL_SAMPLE_PARAMETER_FILE, allow_pickle=False)
                    if STEP_FULL_SAMPLE_PARAMETER_FILE is not None
                    else STEP_FULL_SAMPLE_PARAMETER_VALUES),
    )
    step_full_sample_mu_full = np.load(STEP_FULL_SAMPLE_PARAMETER_FILE, allow_pickle=False) if STEP_FULL_SAMPLE_PARAMETER_FILE is not None else STEP_FULL_SAMPLE_PARAMETER_VALUES
    step_full_sample_weight_matrix = None if STEP_FULL_SAMPLE_WEIGHTS_PATH is None else np.load(STEP_FULL_SAMPLE_WEIGHTS_PATH, allow_pickle=False)
    print("Check: parameter labels must exactly match the order of the loaded snapshots.")
    print("Step viscosity labels: 2e-5 to 2e-3, in the exact VTU folder order.")
elif STEP_FULL_SAMPLE_DATA_SOURCE == "npz":
    with np.load(STEP_FULL_SAMPLE_NPZ_DATA_PATH, allow_pickle=False) as step_full_sample_dataset:
        step_full_sample_fields_all, step_full_sample_mu_full = step_full_sample_dataset["states"], step_full_sample_dataset["parameters"]
        step_full_sample_weight_matrix = step_full_sample_dataset["weights"] if "weights" in step_full_sample_dataset else None
        STEP_FULL_SAMPLE_CASE_IDS = step_full_sample_dataset["case_ids"] if "case_ids" in step_full_sample_dataset else STEP_FULL_SAMPLE_CASE_IDS
    if STEP_FULL_SAMPLE_WEIGHTS_PATH is not None:
        step_full_sample_weight_matrix = np.load(STEP_FULL_SAMPLE_WEIGHTS_PATH, allow_pickle=False)
elif STEP_FULL_SAMPLE_DATA_SOURCE == "synthetic":
    # Explicit lightweight smoke-test configuration; no CFD data is read.
    step_full_sample_fields_all, step_full_sample_mu_full, step_full_sample_weight_matrix = synthetic_bcke_data(n=24, nh=14, seed=step_full_sample_reconstruction_cfg.seed)
    STEP_FULL_SAMPLE_N_TOTAL, STEP_FULL_SAMPLE_N_REPR_TRAIN, STEP_FULL_SAMPLE_N_FINAL_FIT = 24, 16, 20
    STEP_FULL_SAMPLE_CELL2_POLY_CV_ORIGINS, STEP_FULL_SAMPLE_CELL2_POLY_CV_HORIZON = (16,), 4
    step_full_sample_reconstruction_cfg = replace(step_full_sample_reconstruction_cfg, p=2, d=2, max_total_dimension=4, k_start=3)
    step_full_sample_representation_grid = dict(p=(2,), d=(2,), lambda_phi=(step_full_sample_reconstruction_cfg.lambda_phi,),
        lambda_lambda=(step_full_sample_reconstruction_cfg.lambda_lambda,),
        sigma_phi_factor=(step_full_sample_reconstruction_cfg.sigma_phi_factor,),
        sigma_lambda_factor=(step_full_sample_reconstruction_cfg.sigma_lambda_factor,))
    STEP_FULL_SAMPLE_CASE_IDS = None
    print("SYNTHETIC SMOKE TEST: 24 samples, p=2, d=2; not physical step-flow results.")
else:
    raise ValueError("Unknown DATA_SOURCE.")

step_full_sample_mu_full = parameter_array(step_full_sample_mu_full)
STEP_FULL_SAMPLE_MAX_ITERATIONS = STEP_FULL_SAMPLE_QUICK_ITERATIONS if STEP_FULL_SAMPLE_QUICK_TEST else STEP_FULL_SAMPLE_FULL_ITERATIONS
print("States:", step_full_sample_fields_all.shape, "parameters:", step_full_sample_mu_full.shape)
print("W:", "identity (unweighted Euclidean state norm)" if step_full_sample_weight_matrix is None else step_full_sample_weight_matrix.shape)
print("Quick test:", STEP_FULL_SAMPLE_QUICK_TEST, "maximum BCKE iterations:", STEP_FULL_SAMPLE_MAX_ITERATIONS)
display(pd.Series(asdict(step_full_sample_reconstruction_cfg), name="BCKE construction rules"))
display(pd.Series(asdict(step_full_sample_prediction_cfg), name="block KRR and decoder-aware correction"))

step_full_sample_results = run_bcke_experiment_full_sample(
    step_full_sample_fields_all, step_full_sample_mu_full, step_full_sample_weight_matrix, step_full_sample_reconstruction_cfg, step_full_sample_representation_grid, step_full_sample_prediction_cfg,
    n_train=STEP_FULL_SAMPLE_N_REPR_TRAIN, n_final=STEP_FULL_SAMPLE_N_FINAL_FIT, n_total=STEP_FULL_SAMPLE_N_TOTAL, split_mode=STEP_FULL_SAMPLE_SPLIT_MODE,
    refit=STEP_FULL_SAMPLE_REFIT_ON_TRAIN_VALIDATION, max_iterations=STEP_FULL_SAMPLE_MAX_ITERATIONS,
    audit=STEP_FULL_SAMPLE_CHECK_DIRECTIONAL_DERIVATIVES, audit_h=STEP_FULL_SAMPLE_DIRECTIONAL_H,
    audit_tolerance=STEP_FULL_SAMPLE_DIRECTIONAL_TOLERANCE, case_ids=STEP_FULL_SAMPLE_CASE_IDS,
)

# ============================================================================
# E. FINAL FORECAST: VARYING-M DMD a + LOCAL POLYNOMIAL z
# ============================================================================
# step_full_sample_results holds the kernel prediction baseline.
# step_full_sample_cell2_results holds the final DMD and local-polynomial forecast.
add_bcke_relative_errors(step_full_sample_results, step_full_sample_fields_all)
step_full_sample_cell2_begin = time.perf_counter()
step_full_sample_cell2_linear_predictor = LinearCoordinateVaryingM(
    step_full_sample_results["predictor"], window=STEP_FULL_SAMPLE_CELL2_DMD_WINDOW,
    ridge=STEP_FULL_SAMPLE_CELL2_DMD_RIDGE, variation_ridge=STEP_FULL_SAMPLE_CELL2_DMD_VARIATION_RIDGE,
    max_steps=STEP_FULL_SAMPLE_CELL2_DMD_MAX_STEPS,
)
step_full_sample_cell2_decoder_predictor = LocalLatentDecoderContinuation(
    step_full_sample_cell2_linear_predictor, window=STEP_FULL_SAMPLE_CELL2_DECODER_WINDOW,
    ridge=STEP_FULL_SAMPLE_CELL2_DECODER_RIDGE,
    max_horizon_steps=STEP_FULL_SAMPLE_CELL2_DECODER_MAX_HORIZON_STEPS,
)
step_full_sample_cell2_predictor = DirectPolynomialZContinuation(
    step_full_sample_cell2_decoder_predictor, mode="local",
    origins=STEP_FULL_SAMPLE_CELL2_POLY_CV_ORIGINS, horizon=STEP_FULL_SAMPLE_CELL2_POLY_CV_HORIZON,
    degrees=STEP_FULL_SAMPLE_CELL2_POLY_DEGREES, windows=STEP_FULL_SAMPLE_CELL2_POLY_WINDOWS,
    ridges=STEP_FULL_SAMPLE_CELL2_POLY_RIDGES, terminal_weight=STEP_FULL_SAMPLE_CELL2_POLY_TERMINAL_WEIGHT,
)
step_full_sample_cell2_setup_seconds = time.perf_counter()-step_full_sample_cell2_begin
step_full_sample_cell2_begin = time.perf_counter()
step_full_sample_cell2_prediction_all = step_full_sample_cell2_predictor.predict(step_full_sample_results["parameters"])
step_full_sample_cell2_prediction_seconds = time.perf_counter()-step_full_sample_cell2_begin

# Copy mutable reports so the later comparison cells retain their base results.
step_full_sample_cell2_results = dict(step_full_sample_results)
step_full_sample_cell2_results["predictor"] = step_full_sample_cell2_predictor
step_full_sample_cell2_results["prediction_all"] = step_full_sample_cell2_prediction_all
step_full_sample_cell2_results["per_case_errors"] = step_full_sample_results["per_case_errors"].copy()
step_full_sample_cell2_results["per_case_errors"]["prediction_weighted_error"] = np.sqrt(np.maximum(
    weighted_norm_sq_rows(step_full_sample_fields_all-step_full_sample_cell2_prediction_all, step_full_sample_results["problem"].W), 0.0))
add_bcke_relative_errors(step_full_sample_cell2_results, step_full_sample_fields_all)
step_full_sample_cell2_results["metrics"] = step_full_sample_results["metrics"].copy()
step_full_sample_cell2_before_refit = ((step_full_sample_cell2_results["metrics"]["stage"] != "final")
                      & (step_full_sample_cell2_results["metrics"]["mode"] == "parameter-only prediction"))
step_full_sample_cell2_results["metrics"].loc[step_full_sample_cell2_before_refit, "mode"] = "base KRR parameter-only prediction"
for step_full_sample_cell2_split, step_full_sample_cell2_ids in (("final fitting set", step_full_sample_results["final_fit_indices"]),
                              ("test (untouched)", step_full_sample_results["test_indices"])):
    step_full_sample_cell2_metric_mask = ((step_full_sample_cell2_results["metrics"]["stage"] == "final")
                         & (step_full_sample_cell2_results["metrics"]["split"] == step_full_sample_cell2_split)
                         & (step_full_sample_cell2_results["metrics"]["mode"] == "parameter-only prediction"))
    step_full_sample_cell2_metric = state_error(
        step_full_sample_fields_all[step_full_sample_cell2_ids], step_full_sample_cell2_prediction_all[step_full_sample_cell2_ids],
        step_full_sample_results["problem"].mean, step_full_sample_results["problem"].W)
    for step_full_sample_cell2_key, step_full_sample_cell2_value in step_full_sample_cell2_metric.items():
        step_full_sample_cell2_results["metrics"].loc[step_full_sample_cell2_metric_mask, step_full_sample_cell2_key] = step_full_sample_cell2_value
    step_full_sample_cell2_results["metrics"].loc[step_full_sample_cell2_metric_mask, "mode"] = "DMD a + local polynomial z"
step_full_sample_cell2_results["protocol"] = dict(step_full_sample_results["protocol"])
step_full_sample_cell2_results["protocol"]["final_parameter_predictor"] = step_full_sample_cell2_predictor.summary()
step_full_sample_cell2_results["timings"] = dict(step_full_sample_results["timings"])
step_full_sample_cell2_results["timings"]["base_prediction_all_seconds"] = step_full_sample_results["timings"]["prediction_all_seconds"]
step_full_sample_cell2_results["timings"]["local_polynomial_setup_seconds"] = step_full_sample_cell2_setup_seconds
step_full_sample_cell2_results["timings"]["prediction_all_seconds"] = step_full_sample_cell2_prediction_seconds
step_full_sample_cell2_results["timings"]["total_seconds"] += step_full_sample_cell2_setup_seconds+step_full_sample_cell2_prediction_seconds

display(step_full_sample_cell2_results["representation_candidates"])
display(step_full_sample_cell2_results["prediction_candidates"])
display(step_full_sample_cell2_results["derivative_audit"])
display(step_full_sample_cell2_results["prediction_derivative_audit"])
display(pd.Series(step_full_sample_cell2_predictor.summary(), name="final DMD + local polynomial predictor"))
display(step_full_sample_cell2_predictor.cv_results)
display(pd.Series(step_full_sample_cell2_results["problem"].summary(), name="final BCKE fit"))
display(pd.DataFrame(step_full_sample_cell2_results["problem"].bandwidth_report).T)
display(step_full_sample_cell2_results["metrics"])
display(step_full_sample_cell2_results["per_case_errors"].query("original_split == 'test'"))
display(pd.Series(step_full_sample_cell2_results["timings"], name="wall-clock times (seconds)"))

step_full_sample_output_directory = None
if STEP_FULL_SAMPLE_SAVE_OUTPUTS:
    # Reuse the original report writer for the representation and base map,
    # then add the complete active DMD/local-polynomial model components.
    step_full_sample_cell2_export_results = dict(step_full_sample_cell2_results)
    step_full_sample_cell2_export_results["predictor"] = step_full_sample_results["predictor"]
    step_full_sample_output_directory = save_bcke_results(step_full_sample_cell2_export_results, STEP_FULL_SAMPLE_OUTPUT_ROOT)
    (step_full_sample_output_directory/"parameter_model.npz").rename(step_full_sample_output_directory/"base_parameter_model.npz")
    with np.load(step_full_sample_output_directory/"base_parameter_model.npz", allow_pickle=False) as step_full_sample_cell2_archive:
        step_full_sample_cell2_model_arrays = {step_full_sample_key: step_full_sample_cell2_archive[step_full_sample_key].copy() for step_full_sample_key in step_full_sample_cell2_archive.files}
    step_full_sample_cell2_model_arrays.update(
        model_format=np.array("bcke_varying_m_local_poly_v1"),
        linear_G0=step_full_sample_cell2_linear_predictor.G0,
        linear_G1=step_full_sample_cell2_linear_predictor.G1,
        linear_c0=step_full_sample_cell2_linear_predictor.c0,
        linear_a_reference=step_full_sample_cell2_linear_predictor.a_reference,
        linear_anchor_shift=step_full_sample_cell2_linear_predictor.anchor_shift,
        linear_n_pairs=step_full_sample_cell2_linear_predictor.n_pairs,
        linear_max_steps=step_full_sample_cell2_linear_predictor.max_steps,
        mu_min=step_full_sample_cell2_linear_predictor.mu_min,
        mu_boundary=step_full_sample_cell2_linear_predictor.mu_boundary,
        grid_step=step_full_sample_cell2_linear_predictor.grid_step,
        decoder_J=step_full_sample_cell2_decoder_predictor.J,
        decoder_z_boundary=step_full_sample_cell2_decoder_predictor.z_boundary,
        decoder_residual_boundary=step_full_sample_cell2_decoder_predictor.residual_boundary,
        decoder_max_horizon_steps=step_full_sample_cell2_decoder_predictor.max_horizon_steps,
        polynomial_coefficients=step_full_sample_cell2_predictor.z_model.coefficients,
        polynomial_degree=step_full_sample_cell2_predictor.z_model.degree,
        polynomial_mu_scale=step_full_sample_cell2_predictor.z_model.mu_scale,
        polynomial_anchor_shift=step_full_sample_cell2_predictor.anchor_shift,
    )
    np.savez_compressed(step_full_sample_output_directory/"parameter_model.npz", **step_full_sample_cell2_model_arrays)
    step_full_sample_cell2_predictor.cv_results.to_csv(step_full_sample_output_directory/"local_polynomial_cv.csv", index=False)
    step_full_sample_cell2_metadata_path = step_full_sample_output_directory/"settings_and_summary.json"
    step_full_sample_cell2_metadata = json.loads(step_full_sample_cell2_metadata_path.read_text(encoding="utf-8"))
    step_full_sample_cell2_metadata["base_predictor"] = step_full_sample_cell2_metadata["selected_predictor"]
    step_full_sample_cell2_metadata["selected_predictor"] = step_full_sample_cell2_predictor.summary()
    step_full_sample_cell2_metadata["linear_extrapolator"] = step_full_sample_cell2_linear_predictor.summary()
    step_full_sample_cell2_metadata["nonlinear_decoder"] = step_full_sample_cell2_decoder_predictor.summary()
    step_full_sample_cell2_metadata["model_format"] = "bcke_varying_m_local_poly_v1"
    step_full_sample_cell2_metadata["note"] = (
        "Cell 5 final prediction uses varying-M DMD a and direct local-polynomial z. "
        "The fitted interval retains the base map. The original representation and "
        "base KRR selection tables are retained. Local-polynomial CV freezes the "
        "representation and uses only known fitting coordinates. "
        "The test block has previously been viewed; comparisons are exploratory.")
    step_full_sample_cell2_metadata_path.write_text(json.dumps(step_full_sample_cell2_metadata, indent=2, ensure_ascii=False),
                                  encoding="utf-8")

# Plot fitting diagnostics, reconstruction errors, and prediction errors.
plot_bcke_results(step_full_sample_cell2_results, step_full_sample_output_directory, STEP_FULL_SAMPLE_FIGURE_DPI)
plot_bcke_parameter_errors(
    step_full_sample_cell2_results, step_full_sample_output_directory, STEP_FULL_SAMPLE_FIGURE_DPI,
    parameter_axis=STEP_FULL_SAMPLE_PARAMETER_AXIS, parameter_label=STEP_FULL_SAMPLE_PARAMETER_AXIS_LABEL,
)
if step_full_sample_output_directory is not None:
    print("Saved to:", step_full_sample_output_directory.resolve())
print("Final optimizer status:", step_full_sample_cell2_results["problem"].status)
print("Cell 5: backward-facing step; reconstruction on all fitting snapshots.")
print("Forecast: varying-M DMD linear a + direct local-polynomial nonlinear z.")

# Optional parameter-only use after fitting (run only Cells 1 and 2):
# mu_query = np.array([[your_parameter_value]])
# u_query = step_full_sample_cell2_predictor.predict(mu_query)  # shape (n_queries, n_dofs)
# step_full_sample_cell2_results["predictor"] is the same active predictor.
# parameter_model.npz stores its complete components in the new format.
# The legacy predict_saved_bcke loader supports only base_parameter_model.npz.


# ============================================================================
# F. SAME-DATA COMPARISON: BCKE, POD+RBF, AND PPMD FROM SOURCE CELL 4
# ============================================================================
# These parameters match the actual fourth cell of
# PPMD_reconstruction_only_two_splines.ipynb, including its direct high-dimensional
# nonlinear forecast. All three methods use the first 90 cases for fitting.
STEP_FULL_SAMPLE_COMPARISON_POD_ALPHAS = (1e-8, 1e-6, 1e-4, 1e-2, 1.0)
STEP_FULL_SAMPLE_COMPARISON_POD_GAMMAS = (1e-2, 1e-1, 1.0, 10.0, 100.0)
STEP_FULL_SAMPLE_COMPARISON_POD_CV_FOLDS = 5
STEP_FULL_SAMPLE_COMPARISON_PPMD_LINEAR_DIM = 2 if STEP_FULL_SAMPLE_DATA_SOURCE == "synthetic" else 8
STEP_FULL_SAMPLE_COMPARISON_PPMD_NONLINEAR_DIM = 2 if STEP_FULL_SAMPLE_DATA_SOURCE == "synthetic" else 4
STEP_FULL_SAMPLE_COMPARISON_PPMD_CONFIG = dict(
    residual_neighbors=15,
    diffusion_k=22, diffusion_epsilon=100.0, diffusion_alpha=1.6,
    lifting_grid={
        "poly_features__degree": [2],
        "krr__alpha": [1e-4],
        "krr__degree": [3],
        "krr__coef0": [10],
    },
    lifting_cv_folds=10,
    # Each row is (last forecast step, polynomial degree, alpha, coef0).
    linear_schedule=(
        (3, 4, 4e-1, 55),
        (5, 3, 1e-2, 3),
        (7, 3, 8e-2, 10),
        (8, 3, 8e-2, 32000),
        (9, 4, 7, 22),
        (10, 3, 1e-6, 10),
    ),
    nonlinear_grid={
        "trend_degree": [3],
        "trend_alpha": [300],
        "residual_alpha": [1e-5],
        "residual_gamma": [7.8],
    },
    nonlinear_cv_origins=(50, 60, 70, 75, 80),
    nonlinear_cv_horizon=10,
    n_jobs=-1,
)
if STEP_FULL_SAMPLE_DATA_SOURCE == "synthetic":
    # Only the explicit 24-case pipeline test uses shorter CV prefixes.
    STEP_FULL_SAMPLE_COMPARISON_PPMD_CONFIG = dict(STEP_FULL_SAMPLE_COMPARISON_PPMD_CONFIG,
        nonlinear_cv_origins=(10, 12, 16), nonlinear_cv_horizon=4,
        lifting_cv_folds=5, n_jobs=1)

step_full_sample_comparison_mu, step_full_sample_comparison_fit, step_full_sample_comparison_test = _validate_ordered_scalar_comparison(
    step_full_sample_results["parameters"], step_full_sample_results["final_fit_indices"], step_full_sample_results["test_indices"])
step_full_sample_comparison_total_dimension = step_full_sample_cell2_results["problem"].p+step_full_sample_cell2_results["problem"].d
if STEP_FULL_SAMPLE_COMPARISON_PPMD_LINEAR_DIM+STEP_FULL_SAMPLE_COMPARISON_PPMD_NONLINEAR_DIM != step_full_sample_comparison_total_dimension:
    raise ValueError("Set the two PPMD dimensions to the BCKE total dimension for a fair comparison.")

step_full_sample_pod_rbf_prediction_all, step_full_sample_pod_rbf_reference_info = fit_pod_rbf_reference(
    step_full_sample_fields_all, step_full_sample_comparison_mu, step_full_sample_comparison_fit, step_full_sample_comparison_total_dimension,
    step_full_sample_cell2_results["problem"].W, alphas=STEP_FULL_SAMPLE_COMPARISON_POD_ALPHAS,
    gammas=STEP_FULL_SAMPLE_COMPARISON_POD_GAMMAS, cv_folds=STEP_FULL_SAMPLE_COMPARISON_POD_CV_FOLDS,
    seed=step_full_sample_reconstruction_cfg.seed,
)
step_full_sample_pod_rbf_reconstruction_fit = step_full_sample_pod_rbf_reference_info.pop("fitting_reconstruction")
step_full_sample_ppmd_prediction_all, step_full_sample_ppmd_reference_info = fit_ppmd_reference(
    step_full_sample_fields_all, step_full_sample_comparison_mu, step_full_sample_comparison_fit,
    linear_dimension=STEP_FULL_SAMPLE_COMPARISON_PPMD_LINEAR_DIM,
    nonlinear_dimension=STEP_FULL_SAMPLE_COMPARISON_PPMD_NONLINEAR_DIM,
    seed=step_full_sample_reconstruction_cfg.seed, **STEP_FULL_SAMPLE_COMPARISON_PPMD_CONFIG,
)
step_full_sample_comparison_predictions = {
    "BCKE": step_full_sample_cell2_prediction_all,
    "POD+RBF": step_full_sample_pod_rbf_prediction_all,
    "PPMD": step_full_sample_ppmd_prediction_all,
}
# Match source Cell 4's two error regions: known-state reconstruction on the
# fitting prefix and parameter-only prediction on the final test block.
step_full_sample_comparison_estimates = {step_full_sample_name: step_full_sample_estimate.copy()
                        for step_full_sample_name, step_full_sample_estimate in step_full_sample_comparison_predictions.items()}
step_full_sample_comparison_estimates["BCKE"][step_full_sample_comparison_fit] = step_full_sample_cell2_results["reconstruction_all"][step_full_sample_comparison_fit]
step_full_sample_comparison_estimates["POD+RBF"][step_full_sample_comparison_fit] = step_full_sample_pod_rbf_reconstruction_fit
# PPMD's fitting rows already use its diffusion-coordinate lifting reconstruction.
step_full_sample_comparison_errors = build_bcke_comparison_table(
    step_full_sample_fields_all, step_full_sample_comparison_mu, step_full_sample_comparison_estimates,
    step_full_sample_comparison_fit, step_full_sample_comparison_test, step_full_sample_cell2_results["problem"].W,
)
step_full_sample_comparison_errors["evaluation"] = np.where(
    step_full_sample_comparison_errors.region == "fitting", "training reconstruction", "extrapolation prediction")
step_full_sample_comparison_test_errors = step_full_sample_comparison_errors.loc[
    step_full_sample_comparison_errors.region == "test"
].pivot(index="snapshot_index", columns="method", values="relative_weighted_L2")
step_full_sample_comparison_summary = step_full_sample_comparison_errors.groupby(["method", "region"])[
    ["relative_weighted_L2", "absolute_weighted_L2"]
].agg(["mean", "max"])

display(pd.Series(step_full_sample_pod_rbf_reference_info, name="POD+RBF reference"))
display(pd.Series({step_full_sample_key: step_full_sample_value for step_full_sample_key, step_full_sample_value in step_full_sample_ppmd_reference_info.items()
                   if step_full_sample_key != "nonlinear_cv_results"}, name="PPMD source Cell 4 reference"))
display(pd.DataFrame(step_full_sample_ppmd_reference_info["nonlinear_cv_results"]))
display(step_full_sample_comparison_summary)
display(step_full_sample_comparison_test_errors)
plot_bcke_method_comparison(
    step_full_sample_comparison_errors, step_full_sample_comparison_fit, step_full_sample_comparison_test,
    parameter_label=STEP_FULL_SAMPLE_PARAMETER_AXIS_LABEL, output_directory=step_full_sample_output_directory,
    dpi=STEP_FULL_SAMPLE_FIGURE_DPI,
)
if step_full_sample_output_directory is not None:
    step_full_sample_comparison_errors.to_csv(step_full_sample_output_directory/"method_comparison_per_case.csv", index=False)
    step_full_sample_comparison_summary.to_csv(step_full_sample_output_directory/"method_comparison_summary.csv")
    pd.DataFrame(step_full_sample_ppmd_reference_info["nonlinear_cv_results"]).to_csv(
        step_full_sample_output_directory/"ppmd_cell4_nonlinear_cv.csv", index=False)
    with (step_full_sample_output_directory/"comparison_reference_settings.json").open("w", encoding="utf-8") as step_full_sample_comparison_file:
        json.dump(dict(pod_rbf=step_full_sample_pod_rbf_reference_info, ppmd=step_full_sample_ppmd_reference_info),
                  step_full_sample_comparison_file, indent=2, ensure_ascii=False)
    np.savez_compressed(
        step_full_sample_output_directory/"method_comparison_predictions.npz",
        bcke=step_full_sample_cell2_prediction_all, pod_rbf=step_full_sample_pod_rbf_prediction_all,
        ppmd=step_full_sample_ppmd_prediction_all, parameters=step_full_sample_comparison_mu,
        bcke_comparison_estimates=step_full_sample_comparison_estimates["BCKE"],
        pod_rbf_comparison_estimates=step_full_sample_comparison_estimates["POD+RBF"],
        ppmd_comparison_estimates=step_full_sample_comparison_estimates["PPMD"],
        final_fit_indices=step_full_sample_comparison_fit, test_indices=step_full_sample_comparison_test,
    )
print("PPMD parameters and nonlinear forecast follow the supplied notebook's fourth cell.")
print("Comparison figures: training reconstruction and extrapolation prediction, both on log error axes.")
print("All three methods use the same fitting indices, test indices, and weighted error definition.")
print("The final ten test states were viewed in earlier experiments; treat this comparison as exploratory.")


In [ ]:
# Cell 6: step fields and VTUs at the paper comparison parameters.
# Run Cells 1 and 5. Rerun this cell to change the field queries.
import meshio

if "step_full_sample_cell2_results" not in globals():
    raise RuntimeError("Run the step experiment in Cell 5 first.")

STEP_FLOW_DATA_ROOT = STEP_VTU_ROOT
STEP_FLOW_VTU_FILENAME = "backward_facing_step_2d_25.vtu"
STEP_FLOW_COMPONENT = STEP_FULL_SAMPLE_STATE_COMPONENT
STEP_FLOW_RECONSTRUCTION_INDICES = (24,)  # mu=5.0e-4
STEP_FLOW_QUERY_PARAMETERS = (1.01e-3, 1.99e-3)
STEP_FLOW_EXPORT_ALL_TEST = False
STEP_FLOW_SAVE_VTU = True
STEP_FLOW_FIGURE_DPI = STEP_FULL_SAMPLE_FIGURE_DPI
STEP_FLOW_OUTPUT_ROOT = (OUTPUT_DIR / "bcke_step_flow_vtu_outputs")

# Evaluate the already fitted parameter predictor before reading query references.
step_flow_query_mu = parameter_array(np.asarray(STEP_FLOW_QUERY_PARAMETERS, dtype=np.float64)[:, None])
step_flow_query_predictions = step_full_sample_cell2_results["predictor"].predict(step_flow_query_mu)
step_flow_query_paths = [
    resolve_bcke_case_vtu(
        STEP_FLOW_DATA_ROOT, step_flow_parameter, STEP_FLOW_VTU_FILENAME,
        folder_prefix="step", parameter_scale=1e-5)
    for step_flow_parameter in step_flow_query_mu[:, 0]
]
step_flow_query_reference = np.vstack([
    _bcke_vtu_state(meshio.read(step_flow_path), STEP_FLOW_COMPONENT)
    for step_flow_path in step_flow_query_paths
])
if (step_flow_query_predictions.shape != step_flow_query_reference.shape
        or not np.isfinite(step_flow_query_predictions).all()):
    raise ValueError("Query predictions must match the reference VTU state size and be finite.")

# These appended rows are only a display view. The original fitting data,
# representation, predictor, and result dictionary are retained as supplied.
step_flow_original_states = np.asarray(step_full_sample_fields_all, dtype=np.float64)
step_flow_original_mu = parameter_array(step_full_sample_cell2_results["parameters"])
if not np.allclose(
        step_flow_original_mu[np.asarray(STEP_FLOW_RECONSTRUCTION_INDICES), 0],
        5.0e-4, rtol=1e-12, atol=1e-15):
    raise ValueError("The reconstruction row must represent the paper viscosity 5.0e-4.")
step_flow_query_indices = np.arange(
    len(step_flow_original_states),
    len(step_flow_original_states)+len(step_flow_query_mu), dtype=int)
step_flow_display_states = np.vstack((step_flow_original_states, step_flow_query_reference))
step_flow_display_results = dict(step_full_sample_cell2_results)
step_flow_display_results["parameters"] = np.vstack((step_flow_original_mu, step_flow_query_mu))
step_flow_display_results["test_indices"] = step_flow_query_indices
step_flow_display_results["prediction_all"] = np.vstack((
    step_full_sample_cell2_results["prediction_all"], step_flow_query_predictions))
step_flow_display_results["reconstruction_all"] = np.vstack((
    step_full_sample_cell2_results["reconstruction_all"], np.full_like(step_flow_query_reference, np.nan)))

# Use a separate output directory for this paper selection.
step_flow_export = plot_and_export_bcke_case_flow(
    step_flow_display_results, step_flow_display_states,
    data_root=STEP_FLOW_DATA_ROOT, filename=STEP_FLOW_VTU_FILENAME,
    folder_prefix="step", parameter_scale=1e-5, component=STEP_FLOW_COMPONENT,
    reconstruction_indices=STEP_FLOW_RECONSTRUCTION_INDICES,
    selected_test_indices=step_flow_query_indices, query_parameters=(),
    export_all_test=STEP_FLOW_EXPORT_ALL_TEST, save_vtu=STEP_FLOW_SAVE_VTU,
    output_directory=Path(step_full_sample_output_directory)/"paper_parameters",
    output_root=STEP_FLOW_OUTPUT_ROOT, dpi=STEP_FLOW_FIGURE_DPI,
)

# Mark independent query references explicitly in the VTU manifest.
step_flow_manifest = step_flow_export["vtu_manifest"].copy()
if len(step_flow_manifest):
    step_flow_manifest["query_index"] = -1
    step_flow_manifest["parameter_region"] = "fitting reconstruction"
    step_flow_fitting_boundary = float(np.max(
        step_flow_original_mu[step_full_sample_cell2_results["final_fit_indices"], 0]))
    for step_flow_query_number, step_flow_display_index in enumerate(step_flow_query_indices):
        step_flow_mask = ((step_flow_manifest["mode"] == "prediction")
                             & (step_flow_manifest["snapshot_index"] == step_flow_display_index))
        step_flow_manifest.loc[step_flow_mask, "snapshot_index"] = -1
        step_flow_manifest.loc[step_flow_mask, "query_index"] = step_flow_query_number
        step_flow_manifest.loc[step_flow_mask, "parameter_region"] = (
            "interpolation" if step_flow_query_mu[step_flow_query_number, 0] <= step_flow_fitting_boundary
            else "extrapolation")
    step_flow_manifest.to_csv(step_flow_export["output_directory"]/"vtu_manifest.csv", index=False)
    step_flow_export["vtu_manifest"] = step_flow_manifest
    display(step_flow_manifest[["mode", "parameter", "parameter_region", "relative_L2",
                                   "field_vtu", "error_vtu"]])
step_flow_export["prediction_query_parameters"] = step_flow_query_mu[:, 0].copy()
step_flow_export["query_reference_vtus"] = tuple(map(str, step_flow_query_paths))

# All-method fields use the same comparison fits and exact paper queries.
if "step_full_sample_comparison_estimates" not in globals():
    raise RuntimeError("Run the complete step experiment, including its comparison section, first.")
step_flow_pod_rbf_queries = query_pod_rbf_flow(
    step_full_sample_fields_all, step_full_sample_comparison_mu, step_full_sample_comparison_fit,
    step_full_sample_comparison_total_dimension, step_full_sample_cell2_results["problem"].W,
    step_full_sample_pod_rbf_reference_info, step_flow_query_mu,
    grid_prediction=step_full_sample_pod_rbf_prediction_all,
)
step_flow_ppmd_queries = query_ppmd_cell4_flow(
    step_full_sample_fields_all, step_full_sample_comparison_mu, step_full_sample_comparison_fit,
    step_flow_query_mu, step_full_sample_ppmd_reference_info,
    STEP_FULL_SAMPLE_COMPARISON_PPMD_CONFIG, step_full_sample_ppmd_prediction_all,
    seed=step_full_sample_reconstruction_cfg.seed,
)
step_flow_rec_index = int(STEP_FLOW_RECONSTRUCTION_INDICES[0])
step_flow_rec_parameter = float(step_flow_original_mu[step_flow_rec_index, 0])
step_flow_reconstruction_vtu = resolve_bcke_case_vtu(
    STEP_FLOW_DATA_ROOT, step_flow_rec_parameter, STEP_FLOW_VTU_FILENAME,
    folder_prefix="step", parameter_scale=1e-5)
step_flow_method_comparison = plot_and_export_flow_method_comparison(
    step_full_sample_fields_all[step_flow_rec_index],
    {name: estimates[step_flow_rec_index] for name, estimates in step_full_sample_comparison_estimates.items()},
    step_flow_query_reference,
    {"BCKE": step_flow_query_predictions, "POD+RBF": step_flow_pod_rbf_queries, "PPMD": step_flow_ppmd_queries},
    reconstruction_parameter=step_flow_rec_parameter, query_parameters=step_flow_query_mu,
    reconstruction_vtu=step_flow_reconstruction_vtu, query_vtus=step_flow_query_paths,
    component=STEP_FLOW_COMPONENT, case_name="step",
    dimension=step_full_sample_comparison_total_dimension,
    output_directory=step_flow_export["output_directory"],
    dpi=STEP_FLOW_FIGURE_DPI, save_vtu=STEP_FLOW_SAVE_VTU,
)


In [ ]:
# Cell 7: independent step prediction debugging.
# Run Cells 1 and 5 first. Change settings here and rerun only this cell.

from copy import copy

# Linear prediction settings
STEP_DEBUG_DMD_WINDOW = 60
STEP_DEBUG_DMD_RIDGE = 1e-7
STEP_DEBUG_DMD_VARIATION_RIDGE = 1.1e-2
STEP_DEBUG_DMD_MAX_STEPS = 1000

# Nonlinear prediction settings
STEP_TREND_KERNEL_CV_ORIGINS = (50,)
STEP_TREND_KERNEL_CV_HORIZON = 30
STEP_TREND_KERNEL_DEGREES = (1,)
STEP_TREND_KERNEL_WINDOWS = (None,)
STEP_TREND_KERNEL_LENGTHS = (0.4,)
STEP_TREND_KERNEL_RIDGES = (1e-5,)
STEP_TREND_KERNEL_TREND_RIDGE = 1e-6
STEP_TREND_KERNEL_TERMINAL_WEIGHT = 0.0
STEP_TREND_KERNEL_SAVE_OUTPUTS = True
STEP_TREND_KERNEL_OUTPUT_ROOT = (OUTPUT_DIR / "bcke_step_trend_kernel_z_outputs")
STEP_TREND_KERNEL_FIGURE_DPI = 220

if "run_bcke_trend_kernel_prediction_debug" not in globals():
    raise RuntimeError("Run Cell 1 with the prediction debugging functions first.")
if "step_full_sample_cell2_results" not in globals() or "step_full_sample_cell2_decoder_predictor" not in globals():
    raise RuntimeError("Run the step experiment in Cell 5 first.")

# Rebuild the linear predictor so the settings above take effect.
step_debug_linear = LinearCoordinateVaryingM(
    step_full_sample_cell2_decoder_predictor.base,
    window=STEP_DEBUG_DMD_WINDOW,
    ridge=STEP_DEBUG_DMD_RIDGE,
    variation_ridge=STEP_DEBUG_DMD_VARIATION_RIDGE,
    max_steps=STEP_DEBUG_DMD_MAX_STEPS,
)

step_debug_lifting = copy(step_full_sample_cell2_decoder_predictor)
step_debug_lifting.linear = step_debug_linear

step_debug_results = dict(step_full_sample_cell2_results)
step_debug_results["predictor"] = copy(step_full_sample_cell2_results["predictor"])
step_debug_results["predictor"].linear = step_debug_linear
step_debug_results["predictor"].decoder = step_debug_lifting

step_trend_kernel_debug = run_bcke_trend_kernel_prediction_debug(
    step_debug_results, step_full_sample_fields_all,
    step_debug_lifting, case_name="Backward-facing step",
    origins=STEP_TREND_KERNEL_CV_ORIGINS,
    horizon=STEP_TREND_KERNEL_CV_HORIZON,
    degrees=STEP_TREND_KERNEL_DEGREES,
    trend_windows=STEP_TREND_KERNEL_WINDOWS,
    length_scales=STEP_TREND_KERNEL_LENGTHS,
    kernel_ridges=STEP_TREND_KERNEL_RIDGES,
    trend_ridge=STEP_TREND_KERNEL_TREND_RIDGE,
    terminal_weight=STEP_TREND_KERNEL_TERMINAL_WEIGHT,
    save_outputs=STEP_TREND_KERNEL_SAVE_OUTPUTS,
    output_root=STEP_TREND_KERNEL_OUTPUT_ROOT,
    dpi=STEP_TREND_KERNEL_FIGURE_DPI,
)
step_trend_kernel_predictor = step_trend_kernel_debug["predictor"]

# Query only the parameter after fitting:
# u_query = step_trend_kernel_predictor.predict([[your_parameter_value]])
# z_query = step_trend_kernel_predictor.nonlinear_coordinates([[your_parameter_value]])